This notebook details a transformer architecture for exoplanet classification. It was implemented as a project for CSE 576 at the University of Michigan. It was run on colab, using data from the TESS flux curve database.

In [ ]:
import h5py
import torch
from torch.utils.data import Dataset

import torch
import numpy as np
import h5py
from torch.utils.data import Dataset

class TimeSeriesAugmenter:
    """
    Applies random augmentations to time-series data.
    """
    def __init__(self, p_mirror=0.5, p_noise=0.5, noise_std=0.01, max_shift=0.1):
        self.p_mirror = p_mirror
        self.p_noise = p_noise
        self.noise_std = noise_std
        self.max_shift = max_shift

    def __call__(self, sample):
        g_flux = sample["global_flux"].numpy()
        g_cent = sample["global_cent"].numpy()
        l_flux = sample["local_flux"].numpy()
        l_cent = sample["local_cent"].numpy()

        # random mirroring
        if np.random.rand() < self.p_mirror:
            g_flux = np.ascontiguousarray(g_flux[::-1])
            g_cent = np.ascontiguousarray(g_cent[::-1])
            l_flux = np.ascontiguousarray(l_flux[::-1])
            l_cent = np.ascontiguousarray(l_cent[::-1])

        # gaussian noise injection
        if np.random.rand() < self.p_noise:
            g_flux += np.random.normal(0, self.noise_std, g_flux.shape)
            l_flux += np.random.normal(0, self.noise_std, l_flux.shape)

        # global shift
        if self.max_shift > 0:
            shift_int = int(len(g_flux) * self.max_shift)
            shift = np.random.randint(-shift_int, shift_int)
            g_flux = np.roll(g_flux, shift)
            g_cent = np.roll(g_cent, shift)

        return {
            "global_flux": torch.tensor(g_flux, dtype=torch.float32),
            "global_cent": torch.tensor(g_cent, dtype=torch.float32),
            "local_flux":  torch.tensor(l_flux, dtype=torch.float32),
            "local_cent":  torch.tensor(l_cent, dtype=torch.float32)
        }

class TessH5Dataset(Dataset):
    def __init__(self, h5_path, augment=False):
        self.f = h5py.File(h5_path, "r")
        self.global_flux  = self.f["global_flux_view_fluxnorm"]
        self.global_centr = self.f["global_centr_view"]
        self.local_flux   = self.f["local_flux_view_fluxnorm"]
        self.local_centr  = self.f["local_centr_view"]
        self.labels       = self.f["label"]
        self.N = self.labels.shape[0]

        self.augment = augment
        self.augmenter = TimeSeriesAugmenter()

    def __len__(self):
        return self.N

    def __getitem__(self, idx):
        g_flux = self.global_flux[idx]
        g_cent = self.global_centr[idx]
        l_flux = self.local_flux[idx]
        l_cent = self.local_centr[idx]
        label  = self.labels[idx]

        sample = {
            "global_flux": torch.tensor(g_flux, dtype=torch.float32),
            "global_cent": torch.tensor(g_cent, dtype=torch.float32),
            "local_flux":  torch.tensor(l_flux, dtype=torch.float32),
            "local_cent":  torch.tensor(l_cent, dtype=torch.float32)
        }

        if self.augment:
            sample = self.augmenter(sample)

        return sample, torch.tensor(label, dtype=torch.long)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
folder_path = "drive/MyDrive/Exoplanet Detection/"

In [ ]:
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

planet_class_id = 1

train_ds = TessH5Dataset(folder_path + "tess_lightcurves_not_unk.h5", augment=False)

labels = train_ds.labels[:]

N = len(labels)
all_indices = np.arange(N)

# stratified train split
train_idx, temp_idx, _, temp_labels = train_test_split(
    all_indices,
    labels,
    test_size=0.30,
    random_state=42,
    stratify=labels  # <--- KEY
)

# stratified val split
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_ds_split = Subset(train_ds, train_idx)
val_ds = Subset(train_ds, val_idx)
test_ds = Subset(train_ds, test_idx)

train_loader = DataLoader(train_ds_split, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)



In [ ]:
#SPLITTING ONLY FOR AUGMENTATION
# --- SPLITTING LOGIC ---

planet_class_id = 1
h5_file_path = folder_path + "tess_lightcurves_not_unk.h5"

train_ds_base = TessH5Dataset(h5_file_path, augment=True)
eval_ds_base  = TessH5Dataset(h5_file_path, augment=False)

labels = eval_ds_base.labels[:]
N = len(labels)
all_indices = np.arange(N)

train_idx, temp_idx, _, temp_labels = train_test_split(
    all_indices,
    labels,
    test_size=0.30,
    random_state=42,
    stratify=labels
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_ds_split = Subset(train_ds_base, train_idx)

val_ds  = Subset(eval_ds_base, val_idx)
test_ds = Subset(eval_ds_base, test_idx)

train_loader = DataLoader(train_ds_split, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

print(f"Train size: {len(train_ds_split)} (Augmented)")
print(f"Val size:   {len(val_ds)} (Raw)")
print(f"Test size:  {len(test_ds)} (Raw)")

Train size: 40013 (Augmented)
Val size:   8574 (Raw)
Test size:  8575 (Raw)


In [ ]:
train_features, train_labels = next(iter(train_loader))
print(train_labels)

tensor([0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0])


In [ ]:
import math
import torch
import torch.nn as nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=10000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class ConvEmbedding(nn.Module):
    def __init__(self, in_channels=2, hidden=32, d_model=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, hidden, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden, hidden, kernel_size=5, padding=2),
            nn.GELU()
        )
        self.proj = nn.Conv1d(hidden, d_model, kernel_size=1)

    def forward(self, x):
        """
        x = (B, 2, L)
        """
        h = self.conv(x)           # (B, hidden, L)
        h = self.proj(h)           # (B, d_model, L)
        return h.transpose(1, 2)   # (B, L, d_model)


def make_encoder(d_model, heads, ff, layers):
    block = nn.TransformerEncoderLayer(
        d_model=d_model,
        nhead=heads,
        dim_feedforward=ff,
        batch_first=True
    )
    return nn.TransformerEncoder(block, num_layers=layers)


class MultiViewTransitTransformer(nn.Module):
    def __init__(self, d_model=128, heads=8, ff=256, layers=2, num_classes=2):
        super().__init__()

        # local view encoder
        self.local_conv = ConvEmbedding(in_channels=2, d_model=d_model)
        self.local_pos  = PositionalEncoding(d_model)
        self.local_enc  = make_encoder(d_model, heads, ff, layers)

        # global view encoder
        self.global_conv = ConvEmbedding(in_channels=2, d_model=d_model)
        self.global_pos  = PositionalEncoding(d_model)
        self.global_enc  = make_encoder(d_model, heads, ff, layers)

        # final classifier
        self.fc = nn.Sequential(
            nn.Linear(2 * d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, num_classes)
        )

        self.pre_ln = nn.LayerNorm(d_model) #added for exploding gradients

    def encode(self, flux, centr, conv, pos, enc):
        """
        flux, centr: (B, L)
        → returns pooled (B, d_model)
        """
        x = torch.stack([flux, centr], dim=1)  # (B, 2, L)
        h = conv(x)         # (B, L, d_model)
        h = self.pre_ln(h) #added for exploding gradients
        h = pos(h)
        h = enc(h)          # (B, L, d_model)
        h, _ = torch.max(h, dim=1)
        return h

    def forward(self, batch):
        gf = batch["global_flux"]
        gc = batch["global_cent"]
        lf = batch["local_flux"]
        lc = batch["local_cent"]

        h_local = self.encode(lf, lc, self.local_conv, self.local_pos, self.local_enc)
        h_global = self.encode(gf, gc, self.global_conv, self.global_pos, self.global_enc)

        h = torch.cat([h_local, h_global], dim=-1)
        return self.fc(h)


In [ ]:
from sklearn.metrics import accuracy_score, average_precision_score
import torch.nn.functional as F
import torch
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate(model, loader, device, pos_label=None):
    """
    Evaluate on a loader.
    If pos_label is given, also compute PR-AUC for that class.
    """
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds  = []
    all_pos_scores = []  # probability for positive class (for PR-AUC)

    with torch.no_grad():
        for batch_x, batch_y in loader:
            for k in batch_x:
                batch_x[k] = batch_x[k].to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_x)                    # (B, num_classes)
            loss = F.cross_entropy(logits, batch_y)
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)       # (B, num_classes)
            preds = probs.argmax(dim=1)                # (B,)

            all_labels.append(batch_y.cpu().numpy())
            all_preds.append(preds.cpu().numpy())

            if pos_label is not None:
                all_pos_scores.append(probs[:, pos_label].cpu().numpy())

    # concatenate
    import numpy as np
    y_true = np.concatenate(all_labels)
    y_pred = np.concatenate(all_preds)

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(y_true, y_pred)

    pr_auc = None
    if pos_label is not None:
        y_true_bin = (y_true == pos_label).astype(int)
        y_score = np.concatenate(all_pos_scores)
        pr_auc = average_precision_score(y_true_bin, y_score)


    recall = recall_score(y_true, y_pred, average = None)
    precision = precision_score(y_true, y_pred, average = None)
    f1 = f1_score(y_true, y_pred, average = None)

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "pr_auc": pr_auc,
        "recall": recall,
        "precision": precision,
        "f1": f1
    }


In [ ]:
import wandb

run_name = "transformer_12_4_acutally_augment"

run = wandb.init(
    entity="saiavu-university-of-michigan",
    project="exoplanet-detection",
    name = run_name,
    config={
        "architecture": "Transformer",
        "dataset": "known-labels-100",
        "epochs": 100,
        "batch_size": 32,
        "learning_rate": 1e-4,
    },
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saiavu (deepsensor-greatlakes) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import Adam
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

model = MultiViewTransitTransformer(
    d_model=128,
    heads=8,
    ff=256,
    layers=2,
    num_classes=len(np.unique(train_ds.labels[:]))
).to(device)

optimizer = Adam(model.parameters(), lr=1e-5)

EPOCHS = 60

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
        for k in batch_x:
            batch_x[k] = batch_x[k].to(device)

            batch_x[k] = torch.nan_to_num(
                batch_x[k],
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )

            if not torch.isfinite(batch_x[k]).all():
                print(f"[ERROR] Non-finite values in input '{k}' at batch {batch_idx}")
                raise ValueError("Found non-finite values in inputs")
        batch_y = batch_y.to(device)

        logits = model(batch_x)  # (B, num_classes)

        if not torch.isfinite(logits).all():
            print(f"[ERROR] Non-finite logits at batch {batch_idx}")
            print("Logits example:", logits[0])
            raise ValueError("Non-finite logits")

        loss = F.cross_entropy(logits, batch_y)

        if not torch.isfinite(loss):
            print(f"[ERROR] NaN or Inf loss at batch {batch_idx}")
            print("Batch labels:", batch_y)
            print("Logits sample:", logits[0])
            raise ValueError("Non-finite loss")

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        run.log({"batch_loss": loss.item()})

    epoch_loss = total_loss / len(train_loader)

    # validate at end of epoch
    val_metrics = evaluate(
        model,
        val_loader,
        device=device,
        pos_label=planet_class_id,   # needed for PR-AUC
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"epoch_loss={epoch_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | "
        f"val_pr_auc={val_metrics['pr_auc']:.4f}"
    )

    wandb.log({"epoch": epoch + 1, "epoch_loss": epoch_loss, "val_loss": val_metrics['loss'], "val_pr_auc": val_metrics['pr_auc'] })

torch.save(model.state_dict(), run_name + "_model.ckpt")
wandb.save("transformer.ckpt")


Using cuda device
Epoch 1/60 | epoch_loss=0.2382 | val_loss=0.2255 | val_acc=0.9354 | val_pr_auc=0.1741


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 2/60 | epoch_loss=0.2159 | val_loss=0.1995 | val_acc=0.9354 | val_pr_auc=0.2211


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 3/60 | epoch_loss=0.1966 | val_loss=0.1864 | val_acc=0.9354 | val_pr_auc=0.2671


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 4/60 | epoch_loss=0.1843 | val_loss=0.1753 | val_acc=0.9352 | val_pr_auc=0.3179
Epoch 5/60 | epoch_loss=0.1747 | val_loss=0.1684 | val_acc=0.9370 | val_pr_auc=0.3730
Epoch 6/60 | epoch_loss=0.1654 | val_loss=0.1645 | val_acc=0.9353 | val_pr_auc=0.4208
Epoch 7/60 | epoch_loss=0.1579 | val_loss=0.1563 | val_acc=0.9396 | val_pr_auc=0.4626
Epoch 8/60 | epoch_loss=0.1509 | val_loss=0.1452 | val_acc=0.9430 | val_pr_auc=0.4893
Epoch 9/60 | epoch_loss=0.1448 | val_loss=0.1480 | val_acc=0.9392 | val_pr_auc=0.5257
Epoch 10/60 | epoch_loss=0.1382 | val_loss=0.1308 | val_acc=0.9484 | val_pr_auc=0.5638
Epoch 11/60 | epoch_loss=0.1333 | val_loss=0.1258 | val_acc=0.9477 | val_pr_auc=0.6015
Epoch 12/60 | epoch_loss=0.1270 | val_loss=0.1182 | val_acc=0.9519 | val_pr_auc=0.6240
Epoch 13/60 | epoch_loss=0.1232 | val_loss=0.1330 | val_acc=0.9419 | val_pr_auc=0.6202
Epoch 14/60 | epoch_loss=0.1211 | val_loss=0.1109 | val_acc=0.9526 | val_pr_auc=0.6651
Epoch 15/60 | epoch_loss=0.1167 | val_loss=0.1141

[]

In [ ]:
run.finish()

batch_loss,▂▄▃▂▄▇▃▄▂▂▃▃▃█▃▆▂█▂▄▃▂▅▁▂▃█▄▂▃▄▂▁▁▂▆▁▆▇▆
epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇███
epoch_loss,█▇█▆▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▆▅▄▃▃▃▂▃▃▂▂▂▂▂▂▂▂▃▂▂▂▂▁▂▁▂▂▁▁▁▁▁▂▁▁▁▁▁
val_pr_auc,▁▂▃▃▄▄▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇▇███████████
batch_loss,0.01939
epoch,60
epoch_loss,0.08602
val_loss,0.08226
val_pr_auc,0.83444


In [ ]:
torch.save(model.state_dict(), folder_path + "models/" + run_name + "_model.ckpt")

In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

model = MultiViewTransitTransformer(
    d_model=128,
    heads=8,
    ff=256,
    layers=2,
    num_classes=2).to(device)

Using cuda device


In [ ]:
#loaded_model = torch.load(folder_path + "models/" + run_name + "_model.ckpt")
#model.load_state_dict(torch.load(folder_path + "models/" + run_name + "_model.ckpt"))
model.load_state_dict(torch.load(folder_path + "models/transformer_12_2_augment_model.ckpt"))
#model.load_state_dict(torch.load(folder_path + "models/transformer_12_4_acutally_augment_model.ckpt"))


<All keys matched successfully>

In [ ]:
evaluate(model,
        test_loader,
        device=device,
        pos_label=planet_class_id,   # needed for PR-AUC
    )

{'loss': 0.08255039026101516,
 'accuracy': 0.967930029154519,
 'pr_auc': np.float64(0.8255276553147779),
 'recall': 0.8231046931407943,
 'precision': 0.7203791469194313,
 'f1': 0.7683235046335299}

In [ ]:
evaluate(model,
        test_loader,
        device=device,
        pos_label=1,   # needed for PR-AUC
    )

{'loss': 0.09538260825741132,
 'accuracy': 0.9672303206997085,
 'pr_auc': np.float64(0.8222018539010274),
 'recall': array([0.9753148 , 0.85018051]),
 'precision': array([0.98950164, 0.70403587]),
 'f1': array([0.982357  , 0.77023712])}

In [ ]:
len(test_loader)

75

In [ ]:
#print label counts in test_loader
label_counts = {}
for batch_x, batch_y in test_loader:
    for label in batch_y:
        if label.item() not in label_counts:
            label_counts[label.item()] = 0
        label_counts[label.item()] += 1

print(label_counts)
#print ratio of classes
for label in label_counts:
     print(label_counts[label] / (label_counts[0] + label_counts[1]))


{0: 2191, 1: 200}
0.9163529903805939
0.0836470096194061


In [ ]:
/content/drive/MyDrive/Exoplanet Detection/models